In [ ]:
"""
Prepare working GeoPackage for chain logistics (copy + temporary filling costs).

Behavior:
1) verify source file exists,
2) create/overwrite chain_with_cost.gpkg with only resell and filling,
3) set temporary split filling costs in filling layer.

eta for the run 10 minutes
"""

from __future__ import annotations

import sqlite3
from pathlib import Path

import geopandas as gpd
import pandas as pd

DATA_DIR = Path("dataset_big")
SOURCE_GPKG_PATH = DATA_DIR / "full_lpg_chain_nig_3857.gpkg"
OUTPUT_GPKG_PATH = DATA_DIR / "chain_with_cost.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
FILL_COST_OUT_COL = "cost_fil_kg_out"
FILL_COST_IN_COL = "cost_fil_kg_in"
LEGACY_COST_COL = "cost_kg"
TEMP_FILLING_COST_OUT = 0.6
TEMP_FILLING_COST_IN = 0.5

print("[1/4] Checking source GeoPackage...")
if not SOURCE_GPKG_PATH.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_GPKG_PATH}")

print("[2/4] Creating reduced copied GeoPackage (only resell + filling)...")
if OUTPUT_GPKG_PATH.exists():
    OUTPUT_GPKG_PATH.unlink()

with sqlite3.connect(str(SOURCE_GPKG_PATH)) as conn:
    cur = conn.cursor()
    layers = cur.execute(
        "SELECT table_name FROM gpkg_contents WHERE data_type IN ('features', 'attributes')"
    ).fetchall()
    source_layer_names = {name for (name,) in layers}

required_layers = {RESELL_LAYER, FILLING_LAYER}
missing_source_layers = required_layers.difference(source_layer_names)
if missing_source_layers:
    raise KeyError(
        f"Missing required source layer(s): {sorted(missing_source_layers)} in {SOURCE_GPKG_PATH}"
    )

resell_src = gpd.read_file(SOURCE_GPKG_PATH, layer=RESELL_LAYER)
filling_src = gpd.read_file(SOURCE_GPKG_PATH, layer=FILLING_LAYER)
if resell_src.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty in source file.")
if filling_src.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in source file.")

resell_src.to_file(OUTPUT_GPKG_PATH, layer=RESELL_LAYER, driver="GPKG")
filling_src.to_file(OUTPUT_GPKG_PATH, layer=FILLING_LAYER, driver="GPKG")
print(
    f"Created reduced copy: {OUTPUT_GPKG_PATH} | "
    f"resell={len(resell_src):,}, filling={len(filling_src):,}"
)

print("[3/4] Validating layers in reduced copied GeoPackage...")
with sqlite3.connect(str(OUTPUT_GPKG_PATH)) as conn:
    cur = conn.cursor()
    layers = cur.execute(
        "SELECT table_name FROM gpkg_contents WHERE data_type IN ('features', 'attributes')"
    ).fetchall()
    layer_names = {name for (name,) in layers}
    if not layer_names:
        raise RuntimeError(f"No layers found in copied file: {OUTPUT_GPKG_PATH}")
    if FILLING_LAYER not in layer_names:
        raise KeyError(f"Required layer '{FILLING_LAYER}' not found in {OUTPUT_GPKG_PATH}")

print("[4/4] Updating filling-point cost columns in copied file...")
filling_out = gpd.read_file(OUTPUT_GPKG_PATH, layer=FILLING_LAYER)
if filling_out.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in copied file.")

filling_out[FILL_COST_OUT_COL] = float(TEMP_FILLING_COST_OUT)
filling_out[FILL_COST_IN_COL] = float(TEMP_FILLING_COST_IN)
filling_out[FILL_COST_OUT_COL] = pd.to_numeric(filling_out[FILL_COST_OUT_COL], errors="coerce").astype(float)
filling_out[FILL_COST_IN_COL] = pd.to_numeric(filling_out[FILL_COST_IN_COL], errors="coerce").astype(float)

# Keep legacy column for compatibility only.
filling_out[LEGACY_COST_COL] = filling_out[FILL_COST_OUT_COL].astype(float)
filling_out.to_file(OUTPUT_GPKG_PATH, layer=FILLING_LAYER, driver="GPKG")

print(f"Done. Updated {len(filling_out):,} filling rows in copied file.")
print(f"Set {FILL_COST_OUT_COL}={TEMP_FILLING_COST_OUT} and {FILL_COST_IN_COL}={TEMP_FILLING_COST_IN}")
print(f"Source unchanged: {SOURCE_GPKG_PATH}")
print(f"Output file:      {OUTPUT_GPKG_PATH}")

[1/4] Checking source GeoPackage...
[2/4] Creating reduced copied GeoPackage (only resell + filling)...
Created reduced copy: dataset_big\chain_with_cost.gpkg | resell=2,413, filling=375
[3/4] Validating layers in reduced copied GeoPackage...
[4/4] Updating filling-point cost columns in copied file...
Done. Updated 375 filling rows in copied file.
Set cost_fil_kg_out=0.6 and cost_fil_kg_in=0.5
Source unchanged: dataset_big\full_lpg_chain_nig_3857.gpkg
Output file:      dataset_big\chain_with_cost.gpkg


In [2]:
from __future__ import annotations

import sqlite3
import struct
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, dijkstra
from scipy.spatial import cKDTree

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
MOTO_FRICTION_RASTER = DATA_DIR / "friction_moto.tif"

ID_COL = "id_res&fil"
OUT_REF_COL = "filling_reference"
OUT_TIME_COL = "filling_ref_time_min"

CELL_SIZE_METERS = 1000.0
USE_8_NEIGHBORS = False
MIN_CANDIDATES = 3
PRIMARY_SEARCH_RADIUS_KM = 60
MAX_SEARCH_RADIUS_KM = 120
INITIAL_LIMIT_FACTOR = 10.0
FINAL_LIMIT_FACTOR = 16.0
LIMIT_MARGIN_MIN = 30.0
UNASSIGNED_TIME_MIN = 7000.0
NODATA_INT = -1
PROGRESS_EVERY = 200


def _ensure_bytes(blob) -> bytes | None:
    if blob is None:
        return None
    if isinstance(blob, (bytes, bytearray)):
        return bytes(blob)
    if isinstance(blob, memoryview):
        return blob.tobytes()
    return None


def _gpkg_envelope_xy(geom_blob) -> tuple[float | None, float | None, float | None, float | None]:
    b = _ensure_bytes(geom_blob)
    if not b or len(b) < 8:
        return (None, None, None, None)
    if b[0:2] != b"GP":
        return (None, None, None, None)

    flags = b[3]
    envelope_code = (flags >> 1) & 0b111
    endian = "<" if (flags & 0b1) == 1 else ">"

    if envelope_code == 0:
        return (None, None, None, None)

    offset = 8
    env_size_by_code = {1: 32, 2: 48, 3: 48, 4: 64}
    env_size = env_size_by_code.get(envelope_code, 0)
    if env_size == 0 or len(b) < offset + env_size:
        return (None, None, None, None)

    try:
        minx, maxx, miny, maxy = struct.unpack(endian + "4d", b[offset : offset + 32])
        return (float(minx), float(maxx), float(miny), float(maxy))
    except Exception:
        return (None, None, None, None)


def _sqlite_st_isempty(geom_blob):
    b = _ensure_bytes(geom_blob)
    return 1 if (b is None or len(b) == 0) else 0


def _sqlite_st_minx(geom_blob):
    return _gpkg_envelope_xy(geom_blob)[0]


def _sqlite_st_maxx(geom_blob):
    return _gpkg_envelope_xy(geom_blob)[1]


def _sqlite_st_miny(geom_blob):
    return _gpkg_envelope_xy(geom_blob)[2]


def _sqlite_st_maxy(geom_blob):
    return _gpkg_envelope_xy(geom_blob)[3]


def _register_gpkg_sql_functions(conn):
    conn.create_function("ST_IsEmpty", 1, _sqlite_st_isempty)
    conn.create_function("ST_MinX", 1, _sqlite_st_minx)
    conn.create_function("ST_MaxX", 1, _sqlite_st_maxx)
    conn.create_function("ST_MinY", 1, _sqlite_st_miny)
    conn.create_function("ST_MaxY", 1, _sqlite_st_maxy)


def _read_friction(path: Path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        nodata = src.nodata
        profile = src.profile.copy()
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    arr = np.where(arr > 0, arr, np.nan).astype(np.float32)
    return arr, profile


def _map_points_to_grid(gdf: gpd.GeoDataFrame, transform, width: int, height: int):
    rows, cols = rasterio.transform.rowcol(transform, gdf.geometry.x.values, gdf.geometry.y.values)
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)
    inside = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    return rows, cols, inside


def _candidate_idx_adaptive(r: int, c: int, tree: cKDTree, coords: np.ndarray) -> np.ndarray:
    idx = np.array([], dtype=np.int32)

    found_primary = tree.query_ball_point([r, c], r=PRIMARY_SEARCH_RADIUS_KM, p=2)
    if len(found_primary) > 0:
        idx = np.asarray(found_primary, dtype=np.int32)

    if idx.size < MIN_CANDIDATES:
        found_max = tree.query_ball_point([r, c], r=MAX_SEARCH_RADIUS_KM, p=2)
        if len(found_max) > 0:
            idx = np.unique(np.concatenate([idx, np.asarray(found_max, dtype=np.int32)]))

    if idx.size < MIN_CANDIDATES:
        k = min(max(MIN_CANDIDATES, 8), len(coords))
        _, nn = tree.query([r, c], k=k)
        idx = np.unique(np.concatenate([idx, np.atleast_1d(nn).astype(np.int32)]))

    return idx


def _pick_min_time(dist_row: np.ndarray, cand_idx: np.ndarray, fill_nodes: np.ndarray, fill_ids: np.ndarray):
    cand_nodes = fill_nodes[cand_idx]
    t = dist_row[cand_nodes]
    if not np.isfinite(t).any():
        return NODATA_INT, np.nan
    j = int(np.nanargmin(t))
    return int(fill_ids[cand_idx[j]]), float(t[j])


def _run_assign(
    src_node: int,
    r: int,
    c: int,
    graph: csr_matrix,
    friction_min: float,
    base_idx: np.ndarray,
    fill_rows: np.ndarray,
    fill_cols: np.ndarray,
    fill_nodes: np.ndarray,
    fill_ids: np.ndarray,
    src_component: int,
    fill_component: np.ndarray,
):
    cand = np.unique(base_idx)
    if cand.size > 0:
        cand = cand[fill_component[cand] == src_component]

    if cand.size == 0:
        return NODATA_INT, UNASSIGNED_TIME_MIN, "no_candidate_in_component"

    lb = np.hypot(fill_rows[cand] - r, fill_cols[cand] - c) * CELL_SIZE_METERS * max(friction_min, 1e-9)
    limit = float(np.nanmax(lb) * INITIAL_LIMIT_FACTOR + LIMIT_MARGIN_MIN) if lb.size > 0 else np.inf
    dist_row = dijkstra(
        csgraph=graph,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=limit,
    )[0]

    fid, tmin = _pick_min_time(dist_row, cand, fill_nodes, fill_ids)
    if fid >= 0 and np.isfinite(tmin):
        return fid, float(tmin), "base_ok"

    cand_global = np.where(fill_component == src_component)[0].astype(np.int32)
    if cand_global.size == 0:
        return NODATA_INT, UNASSIGNED_TIME_MIN, "no_global_in_component"

    lb2 = np.hypot(fill_rows[cand_global] - r, fill_cols[cand_global] - c) * CELL_SIZE_METERS * max(friction_min, 1e-9)
    limit2 = float(np.nanmax(lb2) * FINAL_LIMIT_FACTOR + LIMIT_MARGIN_MIN) if lb2.size > 0 else np.inf
    dist_row2 = dijkstra(
        csgraph=graph,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=limit2,
    )[0]

    fid2, tmin2 = _pick_min_time(dist_row2, cand_global, fill_nodes, fill_ids)
    if fid2 >= 0 and np.isfinite(tmin2):
        return fid2, float(tmin2), "fallback_ok"

    return NODATA_INT, UNASSIGNED_TIME_MIN, "fallback_unassigned"


t0 = time.time()
print("[1/6] Reading friction raster and building valid mask...")
if not WORK_GPKG.exists():
    raise FileNotFoundError(f"Missing working GeoPackage: {WORK_GPKG}")
if not MOTO_FRICTION_RASTER.exists():
    raise FileNotFoundError(f"Missing raster: {MOTO_FRICTION_RASTER}")

friction, ref_profile = _read_friction(MOTO_FRICTION_RASTER)
height, width = friction.shape
transform = ref_profile["transform"]
crs = ref_profile["crs"]

valid = np.isfinite(friction)
if not np.any(valid):
    raise RuntimeError("No valid cells in motorized friction raster.")
friction_min = float(np.nanpercentile(friction[valid], 5))
print(f"Grid: {width} x {height} | valid cells: {int(valid.sum()):,}")

print("[2/6] Loading reseller and filling points...")
resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(WORK_GPKG, layer=FILLING_LAYER)
if resell.empty:
    raise RuntimeError("Resell layer is empty.")
if filling.empty:
    raise RuntimeError("Filling layer is empty.")

for name, gdf in ((RESELL_LAYER, resell), (FILLING_LAYER, filling)):
    if ID_COL not in gdf.columns:
        raise KeyError(f"Missing '{ID_COL}' in layer '{name}'.")
    if gdf.geometry.isna().all():
        raise RuntimeError(f"All geometries are null in layer '{name}'.")

if resell.crs != crs:
    resell = resell.to_crs(crs)
if filling.crs != crs:
    filling = filling.to_crs(crs)

resell = resell[resell.geometry.notna() & resell.geometry.geom_type.isin(["Point"])].copy()
filling = filling[filling.geometry.notna() & filling.geometry.geom_type.isin(["Point"])].copy()

rid_resell = pd.to_numeric(resell[ID_COL], errors="coerce")
rid_fill = pd.to_numeric(filling[ID_COL], errors="coerce")
if rid_resell.isna().any() or (rid_resell <= 0).any() or (not rid_resell.is_unique):
    raise ValueError(f"Layer '{RESELL_LAYER}' must have unique positive numeric '{ID_COL}'.")
if rid_fill.isna().any() or (rid_fill <= 0).any() or (not rid_fill.is_unique):
    raise ValueError(f"Layer '{FILLING_LAYER}' must have unique positive numeric '{ID_COL}'.")

print("[3/6] Mapping points to friction grid...")
res_rows, res_cols, in_res = _map_points_to_grid(resell, transform, width, height)
fill_rows, fill_cols, in_fill = _map_points_to_grid(filling, transform, width, height)

resell = resell.loc[in_res].copy()
filling = filling.loc[in_fill].copy()
res_rows = res_rows[in_res]
res_cols = res_cols[in_res]
fill_rows = fill_rows[in_fill]
fill_cols = fill_cols[in_fill]
rid_resell = rid_resell.loc[in_res].astype(np.int32).to_numpy()
rid_fill = rid_fill.loc[in_fill].astype(np.int32).to_numpy()

if len(resell) == 0 or len(filling) == 0:
    raise RuntimeError("No points left inside raster extent after mapping.")

print(f"Resellers on grid: {len(resell):,}")
print(f"Fillings on grid : {len(filling):,}")

print("[4/6] Building shortest-path graph on friction_moto...")
node_id = -np.ones((height, width), dtype=np.int32)
vr, vc = np.where(valid)
node_id[vr, vc] = np.arange(len(vr), dtype=np.int32)
n_nodes = len(vr)

if USE_8_NEIGHBORS:
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]
else:
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]

edge_i = []
edge_j = []
edge_c = []
diag_factor = np.sqrt(2.0)

for r, c in zip(vr, vc):
    n0 = node_id[r, c]
    f0 = friction[r, c]
    for dr, dc in neighbors:
        rr = r + dr
        cc = c + dc
        if rr < 0 or rr >= height or cc < 0 or cc >= width:
            continue
        n1 = node_id[rr, cc]
        if n1 < 0:
            continue
        f1 = friction[rr, cc]
        if not np.isfinite(f1):
            continue

        step_m = CELL_SIZE_METERS
        if dr != 0 and dc != 0:
            step_m *= diag_factor

        cost = 0.5 * (f0 + f1) * step_m
        if cost <= 0:
            continue

        edge_i.append(int(n0))
        edge_j.append(int(n1))
        edge_c.append(float(cost))

graph = csr_matrix((edge_c, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
n_comp, comp = connected_components(csgraph=graph, directed=False, return_labels=True)
print(f"Graph nodes: {n_nodes:,} | edges: {len(edge_i):,} | components: {n_comp:,}")

res_nodes = node_id[res_rows, res_cols]
fill_nodes = node_id[fill_rows, fill_cols]

ok_res = res_nodes >= 0
ok_fill = fill_nodes >= 0
res_nodes = res_nodes[ok_res]
res_rows = res_rows[ok_res]
res_cols = res_cols[ok_res]
rid_resell = rid_resell[ok_res]

fill_nodes = fill_nodes[ok_fill]
fill_rows = fill_rows[ok_fill]
fill_cols = fill_cols[ok_fill]
rid_fill = rid_fill[ok_fill]

if len(res_nodes) == 0 or len(fill_nodes) == 0:
    raise RuntimeError("No reseller/filling mapped to valid graph nodes.")

fill_comp = comp[fill_nodes]
fill_tree = cKDTree(np.column_stack([fill_rows, fill_cols]).astype(np.float64))

print("[5/6] Assigning nearest filling per reseller...")
n_res = len(res_nodes)
ref_id = np.full(n_res, NODATA_INT, dtype=np.int32)
ref_time = np.full(n_res, UNASSIGNED_TIME_MIN, dtype=np.float32)
reasons = {}
loop_t0 = time.time()

for k, (src_node, r, c) in enumerate(zip(res_nodes, res_rows, res_cols), start=1):
    base_idx = _candidate_idx_adaptive(int(r), int(c), fill_tree, fill_tree.data)
    src_comp = int(comp[int(src_node)])
    fid, tmin, reason = _run_assign(
        src_node=int(src_node),
        r=int(r),
        c=int(c),
        graph=graph,
        friction_min=friction_min,
        base_idx=base_idx,
        fill_rows=fill_rows,
        fill_cols=fill_cols,
        fill_nodes=fill_nodes,
        fill_ids=rid_fill,
        src_component=src_comp,
        fill_component=fill_comp,
    )
    ref_id[k - 1] = int(fid)
    ref_time[k - 1] = float(tmin)
    reasons[reason] = int(reasons.get(reason, 0)) + 1

    if (k % PROGRESS_EVERY == 0) or (k == n_res):
        elapsed = time.time() - loop_t0
        speed = k / max(elapsed, 1e-9)
        rem = (n_res - k) / max(speed, 1e-9)
        print(f"  {k:,}/{n_res:,} ({100.0*k/n_res:.1f}%) | {speed:.1f} res/s | ETA ~{rem/60:.1f} min")

assigned_ok = (ref_id >= 0) & np.isfinite(ref_time) & (ref_time < UNASSIGNED_TIME_MIN)
print(f"Assigned resellers: {int(assigned_ok.sum()):,}/{n_res:,}")
print("Reason counts:", reasons)

print("[6/6] Updating reseller table in working GeoPackage...")
updates = pd.DataFrame({
    ID_COL: rid_resell,
    OUT_REF_COL: ref_id,
    OUT_TIME_COL: ref_time,
})

with sqlite3.connect(str(WORK_GPKG)) as conn:
    _register_gpkg_sql_functions(conn)
    cur = conn.cursor()
    cols = [c[1] for c in cur.execute(f'PRAGMA table_info("{RESELL_LAYER}")').fetchall()]
    if OUT_REF_COL not in cols:
        cur.execute(f'ALTER TABLE "{RESELL_LAYER}" ADD COLUMN "{OUT_REF_COL}" INTEGER')
    if OUT_TIME_COL not in cols:
        cur.execute(f'ALTER TABLE "{RESELL_LAYER}" ADD COLUMN "{OUT_TIME_COL}" REAL')

    cur.execute(
        f'UPDATE "{RESELL_LAYER}" SET "{OUT_REF_COL}" = ?, "{OUT_TIME_COL}" = ?',
        (int(NODATA_INT), float(UNASSIGNED_TIME_MIN)),
    )

    cur.executemany(
        f'UPDATE "{RESELL_LAYER}" SET "{OUT_REF_COL}" = ?, "{OUT_TIME_COL}" = ? WHERE "{ID_COL}" = ?',
        [
            (int(a), float(t), int(rid))
            for a, t, rid in zip(
                updates[OUT_REF_COL].to_numpy(),
                updates[OUT_TIME_COL].to_numpy(),
                updates[ID_COL].to_numpy(),
            )
        ],
    )
    conn.commit()

print(f"Updated layer: {WORK_GPKG} | {RESELL_LAYER}")
print(f"- Column added/updated: {OUT_REF_COL}")
print(f"- Column added/updated: {OUT_TIME_COL}")
print(f"Done in {(time.time() - t0)/60:.1f} min")

[1/6] Reading friction raster and building valid mask...
Grid: 1442 x 1156 | valid cells: 1,078,258
[2/6] Loading reseller and filling points...
[3/6] Mapping points to friction grid...
Resellers on grid: 2,413
Fillings on grid : 375
[4/6] Building shortest-path graph on friction_moto...
Graph nodes: 1,078,258 | edges: 4,302,586 | components: 20
[5/6] Assigning nearest filling per reseller...
  200/2,413 (8.3%) | 7.4 res/s | ETA ~5.0 min
  400/2,413 (16.6%) | 7.9 res/s | ETA ~4.2 min
  600/2,413 (24.9%) | 5.4 res/s | ETA ~5.6 min
  800/2,413 (33.2%) | 4.5 res/s | ETA ~5.9 min
  1,000/2,413 (41.4%) | 4.1 res/s | ETA ~5.7 min
  1,200/2,413 (49.7%) | 3.7 res/s | ETA ~5.5 min
  1,400/2,413 (58.0%) | 3.6 res/s | ETA ~4.7 min
  1,600/2,413 (66.3%) | 3.1 res/s | ETA ~4.3 min
  1,800/2,413 (74.6%) | 3.1 res/s | ETA ~3.3 min
  2,000/2,413 (82.9%) | 3.2 res/s | ETA ~2.1 min
  2,200/2,413 (91.2%) | 3.2 res/s | ETA ~1.1 min
  2,400/2,413 (99.5%) | 3.1 res/s | ETA ~0.1 min
  2,413/2,413 (100.0%) | 

In [ ]:
"""
Quick QA on reseller -> filling assignment (compact).
"""

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"

resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(WORK_GPKG, layer=FILLING_LAYER)

res_ref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
res_tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")
fill_ids = pd.to_numeric(filling[ID_COL], errors="coerce")
fill_id_set = set(fill_ids[np.isfinite(fill_ids)].astype(np.int64).tolist())

valid_ref = np.isfinite(res_ref) & (res_ref > 0) & pd.Series(res_ref.astype("Int64")).isin(fill_id_set).to_numpy()
valid_time = np.isfinite(res_tmin) & (res_tmin >= 0) & (res_tmin < 7000)
valid = valid_ref & valid_time

n = len(resell)
print("=== QUICK ASSIGNMENT QA ===")
print(f"Valid assignments: {int(valid.sum()):,}/{n:,} ({100.0*valid.mean():.2f}%)")
print(f"Invalid reference: {int((~valid_ref).sum()):,}")
print(f"Invalid time:      {int((~valid_time).sum()):,}")

if int(valid.sum()) > 0:
    t = res_tmin[valid].to_numpy(dtype=float)
    print(
        "Time min/p50/p95/p99/max (min): "
        f"{float(np.nanmin(t)):.2f} / {float(np.nanmedian(t)):.2f} / "
        f"{float(np.nanpercentile(t, 95)):.2f} / {float(np.nanpercentile(t, 99)):.2f} / "
        f"{float(np.nanmax(t)):.2f}"
    )

In [ ]:
"""
Build filling_point_clients from 4.3 output + reseller->filling linkage.
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import pandas as pd

DATA_DIR = Path("dataset_big")
SELL_POINT_GPKG = DATA_DIR / "sell_point_clients.gpkg"
SELL_POINT_LAYER = "sell_point_clients"
CHAIN_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
OUTPUT_GPKG = DATA_DIR / "filling_point_clients.gpkg"
OUTPUT_LAYER = "filling_point_clients"

ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"

CLIENT_COLS = [
    "clients_walk",
    "clients_max_ideal_walk",
    "clients_car",
    "clients_max_ideal_car",
    "clients",
    "clients_max_ideal",
]

print("[1/7] Loading filling base from chain_with_cost...")
filling_44 = gpd.read_file(CHAIN_GPKG, layer=FILLING_LAYER)
if filling_44.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in {CHAIN_GPKG}")
if ID_COL not in filling_44.columns:
    raise KeyError(f"Missing '{ID_COL}' in layer '{FILLING_LAYER}' of {CHAIN_GPKG}")

filling_44[ID_COL] = pd.to_numeric(filling_44[ID_COL], errors="coerce")
filling_44 = filling_44[filling_44[ID_COL].notna() & (filling_44[ID_COL] > 0)].copy()
filling_44[ID_COL] = filling_44[ID_COL].astype("int64")
if filling_44.empty:
    raise RuntimeError("No valid filling IDs found in filling layer.")

filling_base_cols = [c for c in [ID_COL, "place_id", "lat", "lon", "geometry"] if c in filling_44.columns]
filling_base = filling_44[filling_base_cols].copy()
filling_id_set = set(filling_base[ID_COL].tolist())
print(f"Filling points in chain_with_cost: {len(filling_base):,}")

print("[2/7] Loading sell_point_clients from 4.3...")
sell_points = gpd.read_file(SELL_POINT_GPKG, layer=SELL_POINT_LAYER)
if sell_points.empty:
    raise RuntimeError(f"Layer '{SELL_POINT_LAYER}' is empty in {SELL_POINT_GPKG}")

required = [ID_COL] + CLIENT_COLS
missing = [c for c in required if c not in sell_points.columns]
if missing:
    raise KeyError(f"Missing required column(s) in sell_point_clients: {missing}")

sell_points[ID_COL] = pd.to_numeric(sell_points[ID_COL], errors="coerce")
sell_points = sell_points[sell_points[ID_COL].notna() & (sell_points[ID_COL] > 0)].copy()
sell_points[ID_COL] = sell_points[ID_COL].astype("int64")
for c in CLIENT_COLS:
    sell_points[c] = pd.to_numeric(sell_points[c], errors="coerce").fillna(0.0).astype(float)

print("[3/7] Attaching local filling metrics from 4.3 by id_res&fil...")
marker_candidates = ["id_filling_only", "id_fillingonly", "id_fillingOnly"]
marker_col = next((c for c in marker_candidates if c in sell_points.columns), None)

local_cols = [ID_COL] + CLIENT_COLS
for c in ["place_id", "lat", "lon"]:
    if c in sell_points.columns and c not in local_cols:
        local_cols.append(c)
if marker_col is not None and marker_col not in local_cols:
    local_cols.append(marker_col)

local_metrics = sell_points[local_cols].copy()
local_metrics = local_metrics.drop_duplicates(subset=[ID_COL], keep="first")

out = filling_base.merge(local_metrics, on=ID_COL, how="left", suffixes=("", "_sp"))
for c in CLIENT_COLS:
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype(float)
if marker_col is None:
    out["id_filling_only"] = pd.NA
    marker_col = "id_filling_only"

print("[4/7] Aggregating all assigned reseller clients per filling...")
resell_44 = gpd.read_file(CHAIN_GPKG, layer=RESELL_LAYER)
for c in [ID_COL, FILL_REF_COL]:
    if c not in resell_44.columns:
        raise KeyError(f"Missing '{c}' in layer '{RESELL_LAYER}' of {CHAIN_GPKG}")

res_assign = resell_44[[ID_COL, FILL_REF_COL]].copy()
res_assign[ID_COL] = pd.to_numeric(res_assign[ID_COL], errors="coerce")
res_assign[FILL_REF_COL] = pd.to_numeric(res_assign[FILL_REF_COL], errors="coerce")
res_assign = res_assign[
    res_assign[ID_COL].notna()
    & res_assign[FILL_REF_COL].notna()
    & (res_assign[ID_COL] > 0)
    & (res_assign[FILL_REF_COL] > 0)
]
res_assign[ID_COL] = res_assign[ID_COL].astype("int64")
res_assign[FILL_REF_COL] = res_assign[FILL_REF_COL].astype("int64")
res_assign = res_assign[res_assign[FILL_REF_COL].isin(filling_id_set)].copy()

point_clients = sell_points[[ID_COL, "clients", "clients_max_ideal"]].copy()
assigned = res_assign.merge(point_clients, on=ID_COL, how="left")
assigned["clients"] = pd.to_numeric(assigned["clients"], errors="coerce").fillna(0.0).astype(float)
assigned["clients_max_ideal"] = pd.to_numeric(assigned["clients_max_ideal"], errors="coerce").fillna(0.0).astype(float)

agg = (
    assigned.groupby(FILL_REF_COL, as_index=False)[["clients", "clients_max_ideal"]]
    .sum()
    .rename(
        columns={
            FILL_REF_COL: ID_COL,
            "clients": "assigned_fil_clients",
            "clients_max_ideal": "assigned_fil_max_ideal_clients",
        }
    )
)

print("[5/7] Building final filling table...")
out = out.merge(agg, on=ID_COL, how="left")
out["assigned_fil_clients"] = pd.to_numeric(out["assigned_fil_clients"], errors="coerce").fillna(0.0).astype(float)
out["assigned_fil_max_ideal_clients"] = pd.to_numeric(out["assigned_fil_max_ideal_clients"], errors="coerce").fillna(0.0).astype(float)

out["total_fil_clients"] = out["clients"].astype(float) + out["assigned_fil_clients"]
out["total_max_ideal_clients"] = out["clients_max_ideal"].astype(float) + out["assigned_fil_max_ideal_clients"]

keep_cols = [
    ID_COL,
    marker_col,
    "place_id",
    "lat",
    "lon",
    "clients_walk",
    "clients_max_ideal_walk",
    "clients_car",
    "clients_max_ideal_car",
    "clients",
    "clients_max_ideal",
    "total_fil_clients",
    "total_max_ideal_clients",
    "geometry",
]
keep_cols = [c for c in keep_cols if c in out.columns]
out = out[keep_cols].copy()
out = gpd.GeoDataFrame(out, geometry="geometry", crs=filling_44.crs)

print("[6/7] Totals check vs 4.3 output...")
sum_43_clients = float(sell_points["clients"].sum())
sum_43_max = float(sell_points["clients_max_ideal"].sum())
sum_fill_clients = float(out["total_fil_clients"].sum())
sum_fill_max = float(out["total_max_ideal_clients"].sum())

diff_clients = sum_fill_clients - sum_43_clients
diff_max = sum_fill_max - sum_43_max

print("=== TOTALS COMPARISON (filling aggregation vs sell_point_clients) ===")
print(f"4.3 total clients             : {sum_43_clients:,.2f}")
print(f"4.35 total_fil_clients        : {sum_fill_clients:,.2f}")
print(f"Difference                    : {diff_clients:+,.6f}")
print(f"4.3 total clients_max_ideal   : {sum_43_max:,.2f}")
print(f"4.35 total_max_ideal_clients  : {sum_fill_max:,.2f}")
print(f"Difference                    : {diff_max:+,.6f}")

tol = 1e-6
if abs(diff_clients) <= tol and abs(diff_max) <= tol:
    print("Check result: OK (totals match)")
else:
    print(
        "Check result: WARNING (totals differ). "
        "Possible causes: points without valid filling_reference or IDs not aligned across steps."
    )

print("[7/7] Writing filling_point_clients output...")
out.to_file(OUTPUT_GPKG, layer=OUTPUT_LAYER, driver="GPKG")
print(f"Saved: {OUTPUT_GPKG} | layer={OUTPUT_LAYER}")
print(f"Filling rows written: {len(out):,}")

[1/7] Loading filling base from chain_with_cost...
Filling points in chain_with_cost: 375
[2/7] Loading sell_point_clients from 4.3...
[3/7] Attaching local filling metrics from 4.3 by id_res&fil...
[4/7] Aggregating all assigned reseller clients per filling...
[5/7] Building final filling table...
[6/7] Totals check vs 4.3 output...
=== TOTALS COMPARISON (filling aggregation vs sell_point_clients) ===
4.3 total clients             : 26,432,109.99
4.35 total_fil_clients        : 26,432,109.99
Difference                    : +0.000000
4.3 total clients_max_ideal   : 204,794,305.36
4.35 total_max_ideal_clients  : 204,794,305.36
Difference                    : +0.000000
Check result: OK (totals match)
[7/7] Writing filling_point_clients output...
Saved: dataset_big\filling_point_clients.gpkg | layer=filling_point_clients
Filling rows written: 375
